# Sentinelization sanity check

Narrow check of `TraceletCodeAgent._sample_skeleton`: run it on 5 real GAIA questions with a real model and print the generated code alongside its sentinelized skeleton, to eyeball whether the AST-based tool-argument replacement looks right on real model output.

No execution, no fill-in sampling, no judge, no full agent loop -- one model call per example.

In [1]:
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = "gpt-4o"
model = OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])

In [2]:
# Reuse the standard GAIA tool stack, same as naiveReAct.ipynb
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [6]:
from smolagents.tracelet_agent import TraceletCodeAgent

agent = TraceletCodeAgent(
    tools=tools,
    model=model,
    max_steps=50,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    skeleton_strategy="direct_prompt",
)

# agent.run() normally does this before its loop starts (agents.py:491-493) -- since we
# call _sample_skeleton/_execute_candidates directly instead of agent.run(), we have to
# register the tools with the interpreter ourselves, once, or executing any candidate
# code that calls a tool (e.g. web_search) raises InterpreterError: Forbidden function
# evaluation, since the interpreter only allows calls to explicitly registered tools.
agent.python_executor.send_variables(variables=agent.state)
agent.python_executor.send_tools({**agent.tools, **agent.managed_agents})

In [7]:
from common_setup import load_gaia_dataset

eval_ds = load_gaia_dataset(set_to_run="validation")
examples = eval_ds.to_list()[:5]
print(f"Loaded {len(examples)} examples")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 5 examples


In [8]:
# Manually replicate the minimal state agent.run() sets up before the loop (task,
# system prompt, task step) so we can call _sample_skeleton once per question without
# running the full multi-step ReAct loop. Unlike the real loop, there's no retry here,
# so a per-question parsing failure (e.g. the model didn't emit a code block at all) is
# caught and printed rather than stopping us from seeing the rest.
#
# For any skeleton with sentinels: samples n fill-ins, substitutes each, *actually
# executes* each candidate (real tool calls, real side effects -- e.g. a real web search
# per candidate), then runs the judge and prints every candidate's score plus its
# execution result so we can see whether the judge picked the right one.
from smolagents.memory import SystemPromptStep, TaskStep
from smolagents.utils import AgentError
from smolagents.tracelet_agent import ARG_SENTINEL_PREFIX, _parse_scores

for i, example in enumerate(examples):
    question = example["question"]
    agent.task = question
    agent.memory.system_prompt = SystemPromptStep(system_prompt=agent.system_prompt)
    agent.memory.reset()
    agent.memory.steps.append(TaskStep(task=agent.task))

    print("=" * 70)
    print(f"Q{i + 1}: {question[:100]}")
    print("-" * 70)
    try:
        memory_messages = agent.write_memory_to_messages()
        thought, code, code_skeleton, usage = agent._sample_skeleton(memory_messages)
    except AgentError as e:
        print(f"[skipped -- {type(e).__name__}] {e}\n")
        continue

    print(f"Thought:\n{thought}\n")
    print(f"Code:\n{code}\n")
    print(f"Code skeleton:\n{code_skeleton}\n")

    total_input, total_output = usage.input_tokens, usage.output_tokens

    if ARG_SENTINEL_PREFIX not in code_skeleton:
        print("No tool-call arguments to sample -- skeleton has no sentinels.")
    else:
        fillins, fillin_usage = agent._sample_arg_fillins(memory_messages, code_skeleton, agent.n_samples)
        total_input += fillin_usage.input_tokens
        total_output += fillin_usage.output_tokens

        candidates = [agent._substitute(code_skeleton, fillin) for fillin in fillins]
        trials = agent._execute_candidates(candidates)  # real execution, side effects included

        judge_output, judge_usage = agent._score_trials(thought, trials)
        total_input += judge_usage.input_tokens
        total_output += judge_usage.output_tokens
        scores = _parse_scores(judge_output, len(trials))
        winner_index = agent._pick_best(judge_output, len(trials))

        print(f"Fill-ins and judge scores ({len(trials)} candidates):")
        for j, (fillin, (cand_code, observation)) in enumerate(zip(fillins, trials)):
            marker = " <- winner" if j == winner_index else ""
            print(f"  Candidate {j} (score {scores[j]}){marker}:")
            print(f"    Fillin: {fillin}")
            print(f"    Code: {cand_code}")
            print(f"    Observation: {observation}")

    print(f"\nTokens: {total_input} in / {total_output} out")
    print()

Q1: A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure w
----------------------------------------------------------------------
Thought:
Thought: To solve this task, I need to perform the following steps:
1. Search for the AI regulation paper submitted to arXiv.org in June 2022, and find the figure with three axes to identify the label words at both ends of each axis.
2. Search for the Physics and Society article submitted to arXiv.org on August 11, 2016, and check if any of the label words from the 2022 paper are used to describe a type of society.

Let's start by searching for the AI regulation paper submitted to arXiv.org in June 2022.

Code:
ai_regulation_paper = web_search(query=ARG0)
print(ai_regulation_paper)

Code skeleton:
ai_regulation_paper = web_search(query=ARG0)
print(ai_regulation_paper)

Fill-ins and judge scores (3 candidates):
  Candidate 0 (score 0.0):
    Fillin: {'ARG0': '"AI regulation paper arXiv June 2022"'}
    Co

In [ ]:
# Sanity check for the [Tracelet] instrumentation added to _step_stream/_judge_select:
# unlike the piecewise cells above (which call _sample_skeleton/_score_trials/_pick_best
# directly, bypassing the orchestrator), this calls the real _step_stream on a fresh
# question, so its self.logger.log(...) calls should print live below -- confirming
# whether the sampled (fill-in + judge) path or the no-sentinel fallback actually fires.
import time
from smolagents.memory import ActionStep
from smolagents.monitoring import Timing

question = eval_ds.to_list()[5]["question"]
agent.task = question
agent.memory.system_prompt = SystemPromptStep(system_prompt=agent.system_prompt)
agent.memory.reset()
agent.memory.steps.append(TaskStep(task=agent.task))

print(f"Q: {question[:100]}\n")

memory_step = ActionStep(step_number=1, timing=Timing(start_time=time.time()))
outputs = list(agent._step_stream(memory_step))
memory_step.timing.end_time = time.time()

print("\n=== Resulting memory_step ===")
print("model_output (thought):", memory_step.model_output)
print("code_action:", memory_step.code_action)
print("observations:", (memory_step.observations or "")[:300])
print("token_usage:", memory_step.token_usage)
print("is_final_answer:", memory_step.is_final_answer)